# Ontology — brapi-py

Interactive notebook for exploring the BrAPI **Ontology** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Install / upgrade brapi-py — uncomment ONE of the options below:

# Option 1: install from PyPI (once the package is published)
# %pip install -q brapi-py

# Option 2: install from local source (for development / testing)
# import subprocess, sys, pathlib
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e',
#                        str(pathlib.Path('..').resolve())])

import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient
from dotenv import load_dotenv
import os

# Loads credentials from ../.env (never committed to git)
# Create a .env file in the project root with these keys:
#   BRAPI_BASE_URL=https://brapi.example.com
#   BRAPI_TOKEN_ENDPOINT=https://auth.example.com/token
#   BRAPI_CLIENT_ID=my-client
#   BRAPI_CLIENT_SECRET=my-secret
load_dotenv(dotenv_path='../.env')

client = BrapiClient(
    base_url=os.environ['BRAPI_BASE_URL'],
    token_endpoint=os.environ['BRAPI_TOKEN_ENDPOINT'],
    client_id=os.environ['BRAPI_CLIENT_ID'],
    client_secret=os.environ['BRAPI_CLIENT_SECRET'],
)
print('Client ready →', client)

## 3 — List  (`GET /ontologies`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /ontologies — list records
df = (
    client.ontology
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 4 — Single-record CRUD

Get, create, update, and delete individual records.

In [ ]:
# GET /ontologies/ {id}  — fetch one record by primary key
ONTOLOGY_DBID = 'example-001'   # ← change to a real ID

record = client.ontology.get_by_id(ONTOLOGY_DBID)
print(record)

In [ ]:
from brapi.entities.generated_ontology import Ontology

# POST /ontologies — create a new record
new_ontology = Ontology(
    ontologyDbId='',  # assigned by server
    ontologyName='Example Ontology',
)
created = client.ontology.create(new_ontology)
print('Created ontologyDbId:', created.ontologyDbId)

In [ ]:
# PUT /ontologies/ {id} — update the record created above
# (run the create cell first to populate 'created')
created.authors = 'updated value'
updated = client.ontology.update(created.ontologyDbId, created)
print('Updated:', updated.ontologyDbId)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.ontology
    .list()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'authors'
if 'authors' in df.columns:
    df['authors'].value_counts().head(10)
else:
    print('Column not present in this result set')

## 7 — Error handling

In [ ]:
import httpx

# 404 — record not found
try:
    client.ontology.get_by_id('does-not-exist-xyz')
except httpx.HTTPStatusError as e:
    print(f'HTTP {e.response.status_code}: {e.request.url}')


# ValueError from create_many with only one item
try:
    client.ontology.create_many([])
except (ValueError, AttributeError) as e:
    print('Error:', e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')